In [1]:
import numpy as np
import torch
import torch.nn as nn

# ============================================================
# SETTINGS
# ============================================================

# ============================================================
# CONFIG
# ============================================================

DATA_PATH = (
    "/home/maria/SelfStudyThesis/data/"
    "allen_natural_scenes_four_class_composite.npy"
)



# ============================================================
# 1. LOAD DATA
# ============================================================

dat = np.load(DATA_PATH, allow_pickle=True).item()

X = np.asarray(dat["X"], dtype=np.float64).T
labels = np.asarray(dat["stimulus_metadata"]["label"])

# Animal = 1
# Everything else = 0
y = (labels == 0).astype(int)

n, d = X.shape

print("=" * 70)
print("DATA")
print("=" * 70)

print("X:", X.shape)
print("animals:", y.sum())
print("non-animals:", (1 - y).sum())
print("d/n:", d / n)

SEED = 42
HIDDEN_SIZES = [0, 1, 2, 3, 5, 10, 20]
EPOCHS = 500
LR = 1e-2
WEIGHT_DECAY = 1e-3

torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)



# ------------------------------------------------------------
# EXPECTED INPUT:
#
# X: numpy array, shape (118, 40064)
# y: numpy array, shape (118,), values 0/1
#
# Example:
# X = np.load("X.npy")
# y = np.load("y.npy")
# ------------------------------------------------------------

X = X.astype(np.float32)
y = y.astype(np.float32)

print("X:", X.shape)
print("y:", y.shape)
print("animals:", int(y.sum()))
print("non-animals:", int(len(y) - y.sum()))


# ============================================================
# MODEL
# ============================================================

class TinyAnimacyNet(nn.Module):

    def __init__(self, d, hidden):
        super().__init__()

        # hidden=0 = ordinary logistic regression
        if hidden == 0:
            self.net = nn.Linear(d, 1)

        else:
            self.net = nn.Sequential(
                nn.Linear(d, hidden),
                nn.Sigmoid(),
                nn.Linear(hidden, 1)
            )

    def forward(self, x):
        return self.net(x).squeeze(-1)


# ============================================================
# LOO
# ============================================================

def loo(hidden):

    predictions = []
    probabilities = []
    targets = []

    n = len(X)

    print("\n" + "=" * 70)
    print(f"HIDDEN SIZE = {hidden}")
    print("=" * 70)

    for test_idx in range(n):

        train_mask = np.arange(n) != test_idx

        X_train = torch.tensor(
            X[train_mask], device=device
        )

        y_train = torch.tensor(
            y[train_mask], device=device
        )

        X_test = torch.tensor(
            X[test_idx:test_idx+1], device=device
        )

        y_test = int(y[test_idx])

        # ----------------------------------------------------
        # STANDARDIZE USING TRAINING FOLD ONLY
        # ----------------------------------------------------

        mean = X_train.mean(dim=0)
        std = X_train.std(dim=0)

        std = torch.where(std < 1e-6, torch.ones_like(std), std)

        X_train = (X_train - mean) / std
        X_test = (X_test - mean) / std

        # ----------------------------------------------------
        # MODEL
        # ----------------------------------------------------

        torch.manual_seed(SEED + test_idx)

        model = TinyAnimacyNet(
            d=X.shape[1],
            hidden=hidden
        ).to(device)

        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=LR,
            weight_decay=WEIGHT_DECAY
        )

        criterion = nn.BCEWithLogitsLoss()

        # ----------------------------------------------------
        # TRAIN
        # ----------------------------------------------------

        model.train()

        for epoch in range(EPOCHS):

            optimizer.zero_grad()

            logits = model(X_train)

            loss = criterion(logits, y_train)

            loss.backward()

            optimizer.step()

        # ----------------------------------------------------
        # TEST
        # ----------------------------------------------------

        model.eval()

        with torch.no_grad():

            logit = model(X_test)
            prob = torch.sigmoid(logit).item()
            pred = int(prob >= 0.5)

        predictions.append(pred)
        probabilities.append(prob)
        targets.append(y_test)

        running_acc = np.mean(
            np.array(predictions) == np.array(targets)
        )

        print(
            f"\r"
            f"fold {test_idx+1:3d}/{n} | "
            f"loss={loss.item():8.4f} | "
            f"p={prob:6.3f} | "
            f"y={y_test} | "
            f"pred={pred} | "
            f"running acc={running_acc:.4f}",
            end="",
            flush=True
        )

    print()

    return (
        np.array(targets),
        np.array(predictions),
        np.array(probabilities)
    )


# ============================================================
# RUN ARCHITECTURES
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    roc_auc_score
)

results = []

for hidden in HIDDEN_SIZES:

    yt, yp, prob = loo(hidden)

    acc = accuracy_score(yt, yp)
    bal = balanced_accuracy_score(yt, yp)
    auc = roc_auc_score(yt, prob)

    results.append((hidden, acc, bal, auc))

    print()
    print(f"RESULT hidden={hidden}")
    print(f"  Accuracy:          {acc:.4f}")
    print(f"  Balanced accuracy: {bal:.4f}")
    print(f"  AUC:               {auc:.4f}")


# ============================================================
# FINAL TABLE
# ============================================================

print("\n")
print("=" * 70)
print("FINAL RESULTS")
print("=" * 70)

print(f"{'hidden':>8} {'accuracy':>12} {'balanced':>12} {'AUC':>12}")
print("-" * 50)

for hidden, acc, bal, auc in results:
    print(
        f"{hidden:8d} "
        f"{acc:12.4f} "
        f"{bal:12.4f} "
        f"{auc:12.4f}"
    )

best = max(results, key=lambda x: x[1])

print("\nBest by accuracy:")
print(best)

DATA
X: (118, 40064)
animals: 57
non-animals: 61
d/n: 339.52542372881356
Device: cuda
X: (118, 40064)
y: (118,)
animals: 57
non-animals: 61

HIDDEN SIZE = 0
fold 118/118 | loss=  0.0000 | p= 0.000 | y=0 | pred=0 | running acc=0.6780

RESULT hidden=0
  Accuracy:          0.6780
  Balanced accuracy: 0.6776
  AUC:               0.7265

HIDDEN SIZE = 1
fold 118/118 | loss=  0.0993 | p= 0.924 | y=0 | pred=1 | running acc=0.6017

RESULT hidden=1
  Accuracy:          0.6017
  Balanced accuracy: 0.5998
  AUC:               0.6224

HIDDEN SIZE = 2
fold 118/118 | loss=  0.0522 | p= 0.943 | y=0 | pred=1 | running acc=0.6610

RESULT hidden=2
  Accuracy:          0.6610
  Balanced accuracy: 0.6595
  AUC:               0.6773

HIDDEN SIZE = 3
fold 118/118 | loss=  0.0684 | p= 0.352 | y=0 | pred=0 | running acc=0.6441

RESULT hidden=3
  Accuracy:          0.6441
  Balanced accuracy: 0.6431
  AUC:               0.7018

HIDDEN SIZE = 5
fold 118/118 | loss=  0.0120 | p= 0.282 | y=0 | pred=0 | running ac